In [3]:
from __future__ import annotations

import argparse
import json
import os
import re
import unicodedata
from pathlib import Path
from typing import Any

import pytesseract
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
from PIL import ImageOps
from rapidfuzz import fuzz
from pypdf import PdfReader
from tqdm import tqdm


In [4]:
#  importent
from langchain_text_splitters import RecursiveCharacterTextSplitter
PROJECT_ROOT = Path.cwd().parent 

TOC_KEYWORDS = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি")
DEFAULT_MATCH_THRESHOLD = 80
POOPLER_PATH = r"C:\poppler\Library\bin"


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
    separators=["\n\n\n", "\n\n", "\n", "।", " ", ""],
)



In [5]:
#  importent
import json
with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:
        INDEX_INFOS = json.load(file)

<>:3: SyntaxWarning: invalid escape sequence '\A'
<>:3: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Tanvir\AppData\Local\Temp\ipykernel_11632\3291077261.py:3: SyntaxWarning: invalid escape sequence '\A'
  with open("D:\AI-ML\AI Projects\Study RAG\chapters.json", "r", encoding="utf-8") as file:


In [6]:

# def load_chapters(chapters_path: Path, class_no: int, subject_name: str) -> list[dict[str, Any]]:
#     with chapters_path.open("r", encoding="utf-8") as file:
#         index = json.load(file)

#     class_key = f"class_{class_no}"
#     books = index["all_books"][class_key]
#     subject_key = subject_name.casefold()
#     if subject_key == "math":
#         subject_key = "mathematics"
#     if subject_key not in books:
#         available = ", ".join(books)
#         raise ValueError(f"No index for {class_key}/{subject_name}. Available: {available}")

#     chapters = books[subject_key]
#     if not chapters:
#         raise ValueError(f"No chapters found for {class_key}/{subject_key}")
#     return sorted(chapters, key=lambda chapter: chapter["chapter_number"])


In [7]:


# def find_pdf_path(project_root: Path, class_no: int, subject_name: str) -> Path:
#     class_pdf_dir = project_root / "pdf" / f"Class_{class_no}"
#     subject_stem = "Math" if subject_name.casefold() in {"math", "mathematics"} else subject_name
#     pdf_path = next(
#         (
#             candidate
#             for candidate in class_pdf_dir.glob("*.pdf")
#             if candidate.stem.casefold() == subject_stem.casefold()
#         ),
#         class_pdf_dir / f"{subject_stem}.pdf",
#     )
#     if not pdf_path.is_file():
#         raise FileNotFoundError(f"PDF not found for class {class_no}, subject {subject_name}: {pdf_path}")
#     return pdf_path


In [8]:


# def ocr_pdf_to_pages(pdf_path: Path, poppler_path: str | None = None, lang: str = "ben", dpi: int = 300) -> list[dict[str, Any]]:
    
#     convert_options: dict[str, Any] = {"dpi": dpi}
#     if poppler_path:
#         convert_options["poppler_path"] = poppler_path

#     page_count = len(PdfReader(str(pdf_path)).pages)
#     pages = []
#     batch_size = 8
#     with tqdm(total=page_count, desc="OCR processing") as progress:
#         for first_page in range(1, page_count + 1, batch_size):
#             last_page = min(first_page + batch_size - 1, page_count)
#             images = convert_from_path(
#                 str(pdf_path),
#                 first_page=first_page,
#                 last_page=last_page,
#                 **convert_options,
#             )
#             for page_index, image in enumerate(images, start=first_page):
#                 image = ImageOps.autocontrast(image.convert("L"))
#                 text = pytesseract.image_to_string(image, lang=lang)
#                 pages.append({"pdf_page_index": page_index, "text": text})
#                 progress.update(1)
#             images.clear()
#     return pages



In [9]:
#  importent
def get_page_metadatas(class_num, subject, page):
    metas = {}
    if class_num is None or subject is None or page is None or page < 0:
        return metas

    book_section = INDEX_INFOS.get("all_books", {}).get(class_num, {}).get(subject)
    if not book_section:
        return metas

    for chapter in book_section:
        start = chapter.get("starting_page")
        if start is None or start > page:
            continue

        if chapter.get("sub_chapters"):
            for sub in chapter["sub_chapters"]:
                sub_start = sub.get("starting_page")
                if sub_start is not None and sub_start <= page:
                    metas["part"] = chapter.get("chapter_name")
                    metas["chapter"] = sub.get("sub_chapter_name")
                    if sub.get("writer_name"):
                        metas["writer_name"] = sub.get("writer_name")
                    else:
                        metas.pop("writer_name", None)
        else:
            metas["chapter"] = chapter.get("chapter_name")
            metas.pop("part", None)
            metas.pop("writer_name", None)

    return metas
    
print(get_page_metadatas("class_7", "bangla", 80))

{'part': 'কবিতা', 'chapter': 'সাম্য', 'writer_name': 'সুফিয়া কামাল'}


## self written

In [10]:
#  importent
def ocr_pdf_to_pages(pdf_path : Path, poppler_path = POOPLER_PATH, lang='ben'):

    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    keywords=("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")
    tracker = False
    page_num = 0

    page_count = len(PdfReader(str(pdf_path)).pages)
    print(page_count)
    images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
    pages = []

    for i, img in  tqdm(enumerate(images, start=1), total=len(images), desc="Processing images"):
        img = ImageOps.autocontrast(img.convert('L'))
        text = pytesseract.image_to_string(img, lang=lang)

        if any(k in text for k in keywords):
            tracker = True

        if tracker:
            pages.append({"page_number": page_num, "page_content": text, "book_name": sub_name, "class_number": class_number})
            page_num += 1
        else:
            pages.append({"page_number": 0, "page_content": text, "book_name": sub_name, "class_number": class_number})

    return pages

In [11]:
import torch
import torchvision.transforms.functional as F
torch.cuda.is_available()

True

## By GPU Accelaretion

In [12]:

from pathlib import Path
import torch
import torchvision.transforms.functional as F
import easyocr
from pypdf import PdfReader
from pdf2image import convert_from_path
from tqdm import tqdm
from langchain_core.documents import Document

# Initialize the EasyOCR reader outside the function or inside? 
# Better to initialize it once globally to avoid reloading models on every function call.
# This loads both Bengali and English models directly into GPU VRAM.
READER = easyocr.Reader(['bn', 'en'], gpu=torch.cuda.is_available())

def ocr_pdf_to_pages_gpu(pdf_path: Path, poppler_path=POOPLER_PATH):
    class_number = pdf_path.parent.name.lower()
    sub_name = pdf_path.stem.lower()

    # Broadened keywords list to handle potential mixed characters
    keywords = ("সূচিপত্র", "সুচিপত্র", "বিষয়সূচি", "CONTENTS", "contents")
    tracker = False
    page_num = 0

    page_count = len(PdfReader(str(pdf_path)).pages)
    print(f"Total Pages to Process: {page_count}")
    
    # 1. Convert PDF to PIL Images (CPU-bound poppler task)
    images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)
    pages = []

    # Check if GPU is ready
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i, img in tqdm(enumerate(images, start=1), total=len(images), desc=f"Processing images via {device}"):
        
        # 2. Move image to GPU VRAM using PyTorch
        img_tensor = F.to_tensor(img).to(device)  # Normalizes to [0.0, 1.0], Shape: [C, H, W]
        
        # 3. GPU Preprocessing: Convert to Grayscale ('L')
        if img_tensor.shape[0] == 3:  # If RGB
            img_tensor = 0.2989 * img_tensor[0] + 0.5870 * img_tensor[1] + 0.1140 * img_tensor[2]
            img_tensor = img_tensor.unsqueeze(0)  # Shape: [1, H, W]

        # 4. GPU Preprocessing: Autocontrast (Normalize min/max scale)
        min_val, max_val = img_tensor.min(), img_tensor.max()
        if max_val > min_val:
            img_tensor = (img_tensor - min_val) / (max_val - min_val)

        # 5. Convert Tensor back to NumPy uint8 array expected by EasyOCR 
        img_numpy = (img_tensor.squeeze(0).cpu().numpy() * 255).astype('uint8')

        # 6. Run GPU-based OCR Inference
        # paragraph=True merges bounding boxes together into natural text strings
        results = READER.readtext(img_numpy, paragraph=True)
        text = "\n".join([res[1] for res in results])

        # Check for table of contents keywords
        if any(k in text for k in keywords):
            tracker = True

        if tracker:
            pages.append({"page_number": page_num, "page_content": text, "book_name": sub_name, "class_number": class_number})
            page_num += 1
        else:
            pages.append({"page_number": 0, "page_content": text, "book_name": sub_name, "class_number": class_number})

    return pages


In [13]:
len(PdfReader(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_6\BGS.pdf").pages)

70

In [14]:
#  importent
book_dicts = ocr_pdf_to_pages_gpu(Path(r"D:\AI-ML\AI Projects\Study RAG\pdf\Class_6\BGS.pdf"))

documents = []
for item in tqdm(book_dicts, desc="Processing PDF pages"):
    content = item.pop("page_content", "")

    class_num = item.get("class_number")
    subject = item.get("book_name")
    page = item.get("page_number")

    if page is None or page < 0:
        continue  # pre-TOC / cover pages — not indexed

    metas = get_page_metadatas(class_num, subject, page)
    if not metas:
        print(f"⚠️ No metadata found for page {page}")

    doc = Document(page_content=content, metadata=item | metas)
    documents.append(doc)

print(f"{len(documents)} page-level documents")

Total Pages to Process: 70


Processing PDF pages: 100%|██████████| 70/70 [00:00<00:00, 141973.54it/s]

⚠️ No metadata found for page 0
⚠️ No metadata found for page 0
⚠️ No metadata found for page 0
⚠️ No metadata found for page 0
⚠️ No metadata found for page 0
70 page-level documents


In [15]:
documents

[Document(metadata={'page_number': 0, 'book_name': 'bgs', 'class_number': 'class_6'}, page_content='বাংলাদেশ ও বিশ্বপরিচয় ষষ্ঠ শ্রেণি\nতি 5\nজাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড , বাংলাদেশ দাংলাদেশ'),
 Document(metadata={'page_number': 0, 'book_name': 'bgs', 'class_number': 'class_6'}, page_content='জাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড কর্তৃক ২০১২ শিক্ষাবর্ষ থেকে ষষ্ঠ শ্রেণির পাঠ্য পুস্তকরূপে নির্ধারিত\nবাংলাদেশ ও বিশ্বপরিচয় ষষ্ঠ শ্রেণি\n২০২৬ শিক্ষাবরমের জন্য পরিমাজিত\nশ্ঁুঁ টিঁ ^ বলাদেশ\nজাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড , বাংলাদেশ'),
 Document(metadata={'page_number': 0, 'book_name': 'bgs', 'class_number': 'class_6'}, page_content='জাতীয় শিক্ষাক্রম ও পাঠ্যপুস্তক বোর্ড;, বাংলাদেশ ৬৯ ৭০, মতিঝিল বাণিজ্যিক এলাকা, ঢাকা-১০০০ কর্তৃক প্রকাশিত প্রকাশক কর্তৃক সর্বসত্ব সংরক্ষিত ]\nপ্রথম সংস্করণ রচনা ও সম্পাদনা অধ্যাপক ড মুনতাসীর মামুন অধ্যাপক শফিউল আলম আবুল মোমেন অধ্যাপক ড মাহবুব সাদিক অধ্যাপক ড মোরশেদ শফিউল হাসান অধ্যাপক ড সৈয়দ আজিজুল হক সৈয়দ মাহফুজ আলী অধ্যাপক মমতাজউদ্দীন পাটোয়ারী অধ

In [16]:
from collections import defaultdict

def merge_documents_by_chapter(documents: list[Document]) -> list[Document]:
    grouped = defaultdict(list)

    for doc in documents:
        chapter = doc.metadata.get("chapter", "বই তথ্য")
        grouped[chapter].append(doc)

    merged_docs = []
    for chapter, docs in grouped.items():
        # Sort by page_number to keep correct reading order
        docs_sorted = sorted(docs, key=lambda d: d.metadata.get("page_number", 0))

        merged_text = "\n\n".join(d.page_content for d in docs_sorted)
        page_numbers = [d.metadata.get("page_number") for d in docs_sorted]

        merged_docs.append(Document(
            page_content=merged_text,
            metadata={
                "chapter": chapter,
                "book_name": docs_sorted[0].metadata.get("book_name"),
                "class_number": docs_sorted[0].metadata.get("class_number"),
                "page_start": min(page_numbers),
                "page_end": max(page_numbers),
            }
        ))

    return merged_docs

chapter_docs = merge_documents_by_chapter(documents)
for d in chapter_docs:
    print(d.metadata, "-", len(d.page_content), "chars")

{'chapter': 'বই তথ্য', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 0, 'page_end': 0} - 4275 chars
{'chapter': 'সমাজ বিবর্তনের ইতিহাস', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 1, 'page_end': 9} - 15080 chars
{'chapter': 'বাংলাদেশের ইতিহাস', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 10, 'page_end': 18} - 18161 chars
{'chapter': 'বাংলাদেশের সংস্কৃতি ও সমাজ', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 19, 'page_end': 23} - 7503 chars
{'chapter': 'বাংলাদেশের অর্থনীতি', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 24, 'page_end': 33} - 14157 chars
{'chapter': 'বাংলাদেশ ও বাংলাদেশের নাগরিক', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 34, 'page_end': 42} - 13506 chars
{'chapter': 'বাংলাদেশের পরিবেশ', 'book_name': 'bgs', 'class_number': 'class_6', 'page_start': 43, 'page_end': 48} - 8594 chars
{'chapter': 'শিশুর বেড়ে ওঠা ও প্রতিবন্ধকতা: সামাজিকীকরণ', 'book_name': 'bgs', 'class_number': 

In [17]:
final_chunks = text_splitter.split_documents(chapter_docs)


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings

import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


import time
start = time.time()

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    cache_folder=r"D:\AI-ML\AI Projects\Study RAG\model_cache",  # the folder containing models--intfloat--...
    encode_kwargs={"normalize_embeddings": True}
)
print(f"Loaded in {time.time() - start:.2f}s")  # should be a few seconds, not minutes



Loaded in 1.65s


In [19]:
def add_e5_prefix(chunks, prefix="passage: "):
    for chunk in chunks:
        chunk.page_content = prefix + chunk.page_content
    return chunks

final_chunks = add_e5_prefix(final_chunks)  # your chunked Documents from the pipeline

In [20]:
from langchain_chroma import Chroma

persist_directory = "./bangla_books_db"
collection_name = "nctb_books"

if os.path.exists(persist_directory) and os.listdir(persist_directory):
    vectorstore = Chroma(
        persist_directory=persist_directory,
        embedding_function=embeddings,
        collection_name=collection_name
    )
    print("Loaded existing vectorstore.")
else:
    vectorstore = Chroma.from_documents(
        documents=final_chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print(f"Built vectorstore with {len(final_chunks)} chunks.")

Built vectorstore with 164 chunks.


In [21]:
## similarity search

In [24]:
def search(query, k=5):
    results = vectorstore.similarity_search("query: " + query, k=k)
    return results

## BM25 + Dense -> Hybrid Retriver

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(final_chunks)
bm25_retriever.k = 5

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]  # tune based on testing — dense usually weighted slightly higher
)

results = hybrid_retriever.invoke("query: মাছ চাসের জন্য গুরুপ্তপুর্ন পদক্ষেপ কি?")

In [44]:
results

[Document(id='91b10203-e079-422d-9b10-0df4176d7570', metadata={'book_name': 'bgs', 'page_end': 9, 'class_number': 'class_6', 'chapter': 'সমাজ বিবর্তনের ইতিহাস', 'page_start': 1}, page_content='passage: কাজ-: বাংলাদেশের সমাজ যেন মানব সমাজ বিকাশের স্তরসমূহের ধারাবাহিক ফসল"-বিশ্লেষণ করো\nঅনুশীলনী\nবহুনির্বাচনি প্রশ্ন ১ শীতল পাটির জন্য বিখ্যাত অঞ্চল কোনটি? ক রাজশাহী খ খুলনা গ. ফরিদপুর ঘ সিলেট ১ নারীই প্রথম কৃষি কাজ শুরু করেন, কারণ তাদের ছিল - |. সৃজনশীল দৃষ্টিভঙ্গি 11. খাবার সংগ্রহের দায়িত্ব পালন I11. দায়িত্বপালনের বাধ্যবাধকতা নিচের কোনটি সঠিক? ক 4 খ i ও ii\n?\nগ ||1\nঘ. \', 11 ও iii'),
 Document(id='e8f044ff-07fa-46a5-8e55-15ecf4169914', metadata={'book_name': 'bgs', 'class_number': 'class_6', 'chapter': 'বাংলাদেশের ইতিহাস', 'page_end': 18, 'page_start': 10}, page_content="passage: সৃজনশীল প্রশ্ন\nপ্রচুর ধান উতপাদন আকাশপথে আমেরিকায় তৈরি পোশাক রপ্তানি / বিভিন্ন ধরনের দেশীয় ফল উতপাদন পাটজাত দ্রব্য বিশ্বব্যাপী প্রশংসিত তথ্য -১ তথ্য - $ ক মাৎস্যন্যায় কী? খ বাংলা কীভাবে একটি প্রদেশে পরিণত হয়

In [64]:
from torchgen import model
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

parser = StrOutputParser()
prompt = PromptTemplate(template="""You are a bangladeshi teacher helping students to learn fron textual books, 
                        maintain profissionalism and be polite, helpful.
                        student asked query: {query}, give answer by the basis 
                        of the content you have content: {content}. you can use your oun knowladge but 
                        keep that minimum.
                        If no relative content found then replay - that question is not releted or mentioned in textbook,
                        after you may give a short answer by the content with your own knowladge""",
                        input_variables=["query", "content"]
                )

llm = ChatDeepSeek(model= "deepseek-flash", temperature=0.2, max_tokens=2048)

query = "মাছ চাসের জন্য গুরুপ্তপুর্ন পদক্ষেপ কি?"
results = hybrid_retriever.invoke(query)


output = prompt | llm 

In [65]:
result = output.invoke({
    "query": query,
    "content": results
})

In [66]:
print(result.content)

প্রিয় শিক্ষার্থী, তুমি সম্ভবত “মাছ চাষের জন্য গুরুত্বপূর্ণ পদক্ষেপ কী?”—এটি জানতে চেয়েছ।

তোমার দেওয়া পাঠ্যবইয়ের অংশে মাছ চাষের পদক্ষেপ সম্পর্কে সরাসরি কিছু বলা হয়নি। সেখানে শুধু আদিম সমাজে “মাছ ধরার হারপুন” হাতিয়ার হিসেবে এবং গ্রামীণ অর্থনীতিতে “জেলে জাল দিয়ে মাছ ধরছে”—এসব প্রসঙ্গ আছে। এগুলো মূলত মাছ ধরা সম্পর্কিত, মাছ চাষ সম্পর্কিত নয়। তাই এ প্রশ্নের পূর্ণ পাঠ্যবইভিত্তিক উত্তর পাওয়া যায় না।

তবু সাধারণ জ্ঞান থেকে সংক্ষেপে বলা যায়, মাছ চাষের গুরুত্বপূর্ণ পদক্ষেপগুলো হলো—
১. উপযুক্ত পুকুর বা জলাশয় নির্বাচন ও প্রস্তুতি নেওয়া  
২. পানি পরিষ্কার রাখা এবং পানির গুণাগুণ ঠিক রাখা  
৩
